### Phase 1

In [2]:
import numpy as np
import copy

#phase1:setup
#classes, deck setup,turns and states defined

class Card:
    #initializes a card object with a specific color and value
    def __init__(self, color, value):
        self.color = color
        self.value = value
    
    def __repr__(self):
        return f"{self.color} {self.value}"
    #allows direct comparison bw two card objects based on their color and value   
    def __eq__(self, other):
        if isinstance(other, Card):
            return self.color == other.color and self.value == other.value
        return False

def generate_deck():
    colors = ['Red', 'Blue', 'Green', 'Yellow'] 
    values = [str(i) for i in range(10)] + ['Skip'] 
    
    deck_list = []
    for c in colors:
        for v in values:
            new_card = Card(c, v)
            deck_list.append(new_card)
            
    deck = np.array(deck_list)
    np.random.shuffle(deck)
    return deck.tolist()

def initialize_game():
    deck = generate_deck()
    state = {
        'p1_cards': [deck.pop() for _ in range(5)], 
        'p2_cards': [deck.pop() for _ in range(5)], 
        'p3_cards': [deck.pop() for _ in range(5)], 
        'top_card': deck.pop(), 
        'deck': deck,
        'current_turn': 1 
    }
    return state

def get_valid_moves(hand, top_card):
    valid_moves = []
    for c in hand:
        if c.color == top_card.color or c.value == top_card.value:
            valid_moves.append(c)
    return valid_moves

def apply_move(state, player_key, move):
    #Deepcopies the state, processes the card play or draw, and advances the turn (handling Skip logic)
    new_state = copy.deepcopy(state)
    
    if move == 'Draw':
        if new_state['deck']:
            new_state[player_key].append(new_state['deck'].pop())
        new_state['current_turn'] = (new_state['current_turn'] % 3) + 1
    else:
        new_state[player_key].remove(move)
        new_state['top_card'] = move
        if move.value == 'Skip':
            new_state['current_turn'] = (new_state['current_turn'] + 1) % 3 + 1
        else:
            new_state['current_turn'] = (new_state['current_turn'] % 3) + 1
            
    return new_state

def is_terminal(state):
    if len(state['p1_cards']) == 0:
        return True
    elif len(state['p2_cards']) == 0:
        return True
    elif len(state['p3_cards']) == 0:
        return True
    else:
        return False

In [3]:
#partial testing of phase 1
test_state = initialize_game()

print("initial states")
print("Top Card:", test_state['top_card'])
print("Player 1 Hand:", test_state['p1_cards'])
print("Player 2 Hand:", test_state['p2_cards'])
print("Player 3 Hand:", test_state['p3_cards'])

#testing valid moves
p1_hand = test_state['p1_cards']
current_top = test_state['top_card']
p1_moves = get_valid_moves(p1_hand, current_top)
print("\nPlayer 1 Valid Moves:", p1_moves)

if p1_moves:
    chosen_move = p1_moves[0]
else:
    chosen_move = 'Draw'

print("Applying Move:", chosen_move)
new_state = apply_move(test_state, 'p1_cards', chosen_move)
print("\nNew Game State")
print("Turn:", new_state['current_turn'])
print("New Top Card:", new_state['top_card'])
print("Player 1 New Hand:", new_state['p1_cards'])

initial states
Top Card: Red 9
Player 1 Hand: [Blue 1, Blue 5, Yellow 0, Red 2, Green Skip]
Player 2 Hand: [Blue Skip, Green 0, Green 4, Yellow 8, Green 3]
Player 3 Hand: [Yellow 2, Red 6, Green 2, Blue 0, Red 1]

Player 1 Valid Moves: [Red 2]
Applying Move: Red 2

New Game State
Turn: 2
New Top Card: Red 2
Player 1 New Hand: [Blue 1, Blue 5, Yellow 0, Green Skip]


### Phase 2

In [4]:
#phase 2: eval function
#adjusts weights etc here
#also drew handwritten part here as well as the explanation of functions in markdown

def get_opponent_keys(player_key):
    players = ['p1_cards', 'p2_cards', 'p3_cards']
    opponents = []
    for p in players:
        if p != player_key:
            opponents.append(p)
    return opponents

def base_evaluation(state, player_key, weights):
    ai_hand = state[player_key]
    opp_keys = get_opponent_keys(player_key)
    c_ai = len(ai_hand)
    
    #calculates average opponent cards
    opp1_key = opp_keys[0]
    opp2_key = opp_keys[1]
    opp1_count = len(state[opp1_key])
    opp2_count = len(state[opp2_key])
    c_opp = (opp1_count + opp2_count) / 2
    s_cards = 0
    for card in ai_hand:
        if card.value == 'Skip':
            s_cards = s_cards + 1

    # score = 50 + (w1 * c_ai) + (w2 * c_opp) + (w3 * s_cards)
    score = 50
    score = score + (weights[0] * c_ai)
    score = score + (weights[1] * c_opp)
    score = score + (weights[2] * s_cards)
    
    return score

def eval_defensive(state, player_key):
    weights = [-5, 2, 3]
    score = base_evaluation(state, player_key, weights)
    return score

def eval_offensive(state, player_key):
    weights = [-8, 1, 1]
    score = base_evaluation(state, player_key, weights)
    return score

In [5]:
#partial testing of phase 2
test_state = initialize_game()

print("hands for eval")
print("Player 1 Hand (Focus AI):", test_state['p1_cards'])
print("Player 2 Hand (Opponent):", test_state['p2_cards'])
print("Player 3 Hand (Opponent):", test_state['p3_cards'])

#counts skips for P1 manually
p1_hand = test_state['p1_cards']
p1_skips = 0
for card in p1_hand:
    if card.value == 'Skip':
        p1_skips = p1_skips + 1

print("\nPlayer 1 currently holds", p1_skips, "'Skip' cards.")
defensive_score = eval_defensive(test_state, 'p1_cards')
offensive_score = eval_offensive(test_state, 'p1_cards')
print("\ncalculated AI Scores for Player 1")
print("Defensive Score [-5, 2, 3]:", defensive_score)
print("Offensive Score [-8, 1, 1]:", offensive_score)

hands for eval
Player 1 Hand (Focus AI): [Blue 7, Blue 1, Red 9, Green 5, Yellow 7]
Player 2 Hand (Opponent): [Blue Skip, Blue 3, Yellow 1, Blue 2, Red 6]
Player 3 Hand (Opponent): [Yellow 9, Yellow Skip, Yellow 2, Red 5, Red 1]

Player 1 currently holds 0 'Skip' cards.

calculated AI Scores for Player 1
Defensive Score [-5, 2, 3]: 35.0
Offensive Score [-8, 1, 1]: 15.0


### Evaluation Functions Explanation

The AI uses a simple formula to score how "good" its current hand is. A higher score means the AI is in a better position to win. 

Formula: Score = 50 - (AI Cards * W1) + (Opponent Cards * W2) + (Skip Cards * W3)

tuned the weights (W) differently for two play styles:

1. Defensive Mode (Weights: -5, 2, 3):** Focuses on survival. It highly values hoarding "Skip" cards to block opponents and prefers opponents to have lots of cards.
2. Offensive Mode (Weights: -8, 1, 1):** Focuses on attacking. It heavily penalizes holding *any* cards, forcing the AI to play its hand aggressively and reach zero cards as fast as possible.

### Phase 3

In [6]:
#phase 3: minimax and expectimax algos
#also adjusted handwritten tree while this phase that i started in phase 2
def minimax(state, depth, maximizing_player, player_key):
    is_done = is_terminal(state)
    if depth == 0 or is_done:
        score = eval_defensive(state, player_key)
        return score, None

    valid_moves = get_valid_moves(state[player_key], state['top_card'])
    if not valid_moves:
        valid_moves = ['Draw']

    if maximizing_player:
        max_eval = float('-inf')
        best_move = valid_moves[0]
        for move in valid_moves:
            new_state = apply_move(state, player_key, move)
            results = minimax(new_state, depth - 1, False, player_key)
            eval_score = results[0]
            
            if eval_score > max_eval:
                max_eval = eval_score
                best_move = move
        return max_eval, best_move
        
    else:
        min_eval = float('inf')
        best_move = valid_moves[0]
        for move in valid_moves:
            new_state = apply_move(state, player_key, move)
            results = minimax(new_state, depth - 1, True, player_key)
            eval_score = results[0]
            
            if eval_score < min_eval:
                min_eval = eval_score
                best_move = move
        return min_eval, best_move

def expectimax(state, depth, is_ai_turn, player_key):
    is_done = is_terminal(state)
    if depth == 0 or is_done:
        score = eval_offensive(state, player_key)
        return score, None

    valid_moves = get_valid_moves(state[player_key], state['top_card'])

    if is_ai_turn: 
        #when ai has to draw card
        if not valid_moves:
            expected_value = 0
            deck_cards = state['deck']
            if deck_cards:
                deck_length = len(deck_cards)
                prob = 1 / deck_length
                for card in deck_cards:
                    new_state = apply_move(state, player_key, 'Draw')
                    #here we set drawn card specifically
                    new_state[player_key][-1] = card 
                    results = expectimax(new_state, depth - 1, False, player_key)
                    eval_score = results[0]
                    expected_value = expected_value + (prob * eval_score)
            return expected_value, 'Draw'
            
        #maximizing logic
        max_eval = float('-inf')
        best_move = valid_moves[0]
        for move in valid_moves:
            new_state = apply_move(state, player_key, move)
            results = expectimax(new_state, depth - 1, False, player_key)
            eval_score = results[0]
            if eval_score > max_eval:
                max_eval = eval_score
                best_move = move
        return max_eval, best_move
        
    else:
        if valid_moves:
            move = np.random.choice(valid_moves)
        else:
            move = 'Draw'
            
        new_state = apply_move(state, player_key, move)
        results = expectimax(new_state, depth - 1, True, player_key)
        eval_score = results[0]
        return eval_score, move

In [7]:
test_state = initialize_game()
print("current top card and cards that AI has")
top_card = test_state['top_card']
p1_hand = test_state['p1_cards']
p2_hand = test_state['p2_cards']

print("top Card:", top_card)
print("P1 Hand (Def):", p1_hand)
print("P2 Hand (Off):", p2_hand)
print("\nTesting P1 Minimax (Depth 2)")

#minimax
p1_results = minimax(test_state, 2, True, 'p1_cards')
p1_score = p1_results[0]
p1_move = p1_results[1]
print("Minimax chooses:", p1_move)
print("Expected Score:", p1_score)

print("\ntesting P2 Expectimax (Depth 2)")
#expectimax
p2_results = expectimax(test_state, 2, True, 'p2_cards')
p2_score = p2_results[0]
p2_move = p2_results[1]
print("Expectimax chooses:", p2_move)
print("Expected Score:", p2_score)

current top card and cards that AI has
top Card: Red 2
P1 Hand (Def): [Red 7, Yellow 4, Red 4, Red 0, Blue 0]
P2 Hand (Off): [Green 7, Red 6, Green 2, Yellow 8, Blue Skip]

Testing P1 Minimax (Depth 2)
Minimax chooses: Red 7
Expected Score: 45.0

testing P2 Expectimax (Depth 2)
Expectimax chooses: Green 2
Expected Score: 32.0


### Final Phase(GUI via tkinter-BONUS)
#### Added Audio as well so kindly load audio in same folder for the complete immersive game experience
#### NOTE: kindly install 'pygame' library for audio effects

In [8]:
import pygame 
import os
import tkinter as tk
from tkinter import scrolledtext

#search tree and game tree generation here
def generate_example_text_tree(state, player_key):
    print("->GENERATED SEARCH TREE(example)")
    top_card = state['top_card']
    hand = state[player_key]
    print("Top card:", top_card)
    print("AI hand:")
    for card in hand:
        print(card)
    print("->GENERATED Game Tree sim(depth=1)")
    print("\nAI decision(All possible decisions considered at this depth=1):")
    valid_moves = get_valid_moves(hand, top_card)
    if valid_moves:
        moves_to_eval = valid_moves
    else:
        moves_to_eval = ['Draw']
        
    num_moves = len(moves_to_eval)
    #formatted same as in doc
    print("\n          AI (Root)")
    if num_moves == 1:
        print("              |")
    elif num_moves == 2:
        print("         /          \\")
    elif num_moves == 3:
        print("       /       |       \\")
    else:
        print("   /      |       |      \\") 
        
    #build move names list manually
    move_names = []
    for m in moves_to_eval:
        move_names.append(str(m))
    
    for name in move_names:
        print(name, "  ", end="")
    print("") # New line
    
    if num_moves == 1:
        print("              |")
    elif num_moves == 2:
        print("         /          \\")
    elif num_moves == 3:
        print("       /       |       \\")
    else:
        print("   /      |       |      \\")
        
    node_types = []
    for move in moves_to_eval:
        if move == 'Draw':
            node_types.append("Chance")
        else:
            node_types.append("Opponent")
            
    for n in node_types:
        print(n, "  ", end="")
    print("\n")
    
    for move in moves_to_eval:
        new_state = apply_move(state, player_key, move)
        #results from minimax
        results = minimax(new_state, 2, False, player_key)
        score = results[0]
        print("Play:", move)
        print("Expected score:", score)
    print("-\n")

#actuall game simulation
class UnoSimulationApp:
    def __init__(self, root, initial_state):
        self.root = root
        self.root.title("GAME AI – UNO by Wasil Kayani(24i-2551)")
        sw = self.root.winfo_screenwidth()
        sh = self.root.winfo_screenheight()
        geometry_string = str(sw) + "x" + str(sh)
        self.root.geometry(geometry_string)
        
        try:
            self.root.state('zoomed') 
        except:
            self.root.attributes('-fullscreen', True)
            
        self.root.configure(bg="#1B2631") 
        self.state = initial_state
        self.p3_mode = "SIMULATION" 
        self.waiting_for_user = False
        
        pygame.mixer.init()
        self.sounds = {}
        self.audio_enabled = False
        try:
            #ADDED THESE SOUNDS THAT ARE TO BE PLAYED VIA 'pygame' library
            #SOUND .wav files attached in zip file
            self.sounds['start'] = pygame.mixer.Sound("start.wav")
            self.sounds['play'] = pygame.mixer.Sound("play.wav")
            self.sounds['draw'] = pygame.mixer.Sound("draw.wav")
            self.sounds['uno'] = pygame.mixer.Sound("uno.wav")
            self.sounds['win'] = pygame.mixer.Sound("win.wav")
            self.audio_enabled = True
        except:
            print("Audio missing, game running without audio")

        #used tkinter's framing concept to add in menu screen as well
        #done via help from ai
        self.menu_frame = tk.Frame(self.root, bg="#1B2631")
        self.game_frame = tk.Frame(self.root, bg="#2E4053")
        
        self.build_menu_ui()
        self.build_game_ui()
        
        self.menu_frame.pack(fill=tk.BOTH, expand=True)
        self.play_audio('start', True)

    def build_menu_ui(self):
        tk.Label(self.menu_frame, text="GAME AI-UNO", font=('Impact', 80), bg="#1B2631", fg="#E74C3C").pack(pady=(120, 0))
        tk.Label(self.menu_frame, text="by Wasil Kayani 24i-2551", font=('Arial', 30, 'bold'), bg="#1B2631", fg="#F1C40F").pack(pady=(0, 40))
        
        tk.Label(self.menu_frame, text="Player 1: Defensive Minimax  |  Player 2: Offensive Expectimax", font=('Arial', 14), bg="#1B2631", fg="#AEDBCE").pack(pady=10)
                 
        sim_btn = tk.Button(self.menu_frame, text="START (SIMULATION MODE)\nWatch 3 AI play", command=lambda: self.start_game("SIMULATION"), 
                              font=('Arial', 16, 'bold'), bg="#2ECC71", fg="white", padx=20, pady=10)
        sim_btn.pack(pady=20)
        
        man_btn = tk.Button(self.menu_frame, text="START (MANUAL MODE)\nYou play as Player 3", command=lambda: self.start_game("MANUAL"), 
                              font=('Arial', 16, 'bold'), bg="#3498DB", fg="white", padx=20, pady=10)
        man_btn.pack(pady=10)

    def start_game(self, mode):
        self.p3_mode = mode
        if self.audio_enabled:
            self.sounds['start'].fadeout(1000) 
            self.play_audio('draw')
            
        self.menu_frame.pack_forget()
        self.game_frame.pack(fill=tk.BOTH, expand=True)
        
        if mode == "SIMULATION":
            mode_text = "AI Simulation Mode"
        else:
            mode_text = "Manual Mode (You are P3)"
            
        self.log("Game started! [" + mode_text + "] Click 'Play Next Turn' to start.")
        self.update_ui()

    def build_game_ui(self):
        self.table_frame = tk.Frame(self.game_frame, bg="#2E4053")
        self.table_frame.pack(fill=tk.BOTH, expand=True, pady=10)
        
        self.table_frame.grid_columnconfigure(0, weight=1)
        self.table_frame.grid_columnconfigure(1, weight=1)
        self.table_frame.grid_columnconfigure(2, weight=1)
        self.table_frame.grid_rowconfigure(0, weight=1)
        self.table_frame.grid_rowconfigure(1, weight=1)
        self.table_frame.grid_rowconfigure(2, weight=1)
        self.table_frame.grid_rowconfigure(3, weight=1)
        
        self.p1_indicator = tk.Label(self.table_frame, text="", bg="#2E4053", fg="#F1C40F", font=('Arial', 14, 'bold'))
        self.p1_indicator.grid(row=0, column=1, sticky="s")
        self.p1_frame = tk.LabelFrame(self.table_frame, text="Player 1 (Defensive)", bg="#2E4053", fg="white", font=('Arial', 12, 'bold'), bd=0)
        self.p1_frame.grid(row=1, column=1, pady=5)
        
        self.p2_indicator = tk.Label(self.table_frame, text="", bg="#2E4053", fg="#F1C40F", font=('Arial', 14, 'bold'))
        self.p2_indicator.grid(row=2, column=0, sticky="s")
        self.p2_frame = tk.LabelFrame(self.table_frame, text="Player 2 (Offensive)", bg="#2E4053", fg="white", font=('Arial', 12, 'bold'), bd=0)
        self.p2_frame.grid(row=3, column=0, pady=5)
        
        self.center_frame = tk.Frame(self.table_frame, bg="#2E4053")
        self.center_frame.grid(row=3, column=1, pady=10)
        
        self.uno_alert = tk.Label(self.center_frame, text="", bg="#2E4053", fg="#E74C3C", font=('Impact', 30))
        self.uno_alert.pack()
        
        tk.Label(self.center_frame, text="TOP CARD", bg="#2E4053", fg="white", font=('Arial', 14, 'bold')).pack(pady=(0,10))
        self.top_card_container = tk.Frame(self.center_frame, bg="#2E4053")
        self.top_card_container.pack()
        
        self.p3_indicator = tk.Label(self.table_frame, text="", bg="#2E4053", fg="#F1C40F", font=('Arial', 14, 'bold'))
        self.p3_indicator.grid(row=2, column=2, sticky="s")
        self.p3_frame = tk.LabelFrame(self.table_frame, text="Player 3", bg="#2E4053", fg="white", font=('Arial', 12, 'bold'), bd=0)
        self.p3_frame.grid(row=3, column=2, pady=5)

        self.control_frame = tk.Frame(self.game_frame, bg="#1B2631")
        self.control_frame.pack(fill=tk.X, side=tk.BOTTOM)
        
        self.next_btn = tk.Button(self.control_frame, text="Play Next Turn", command=self.play_turn, font=('Arial', 14, 'bold'), bg="#F39C12", fg="black", padx=20, pady=10)
        self.next_btn.pack(pady=10)
        
        self.text_area = scrolledtext.ScrolledText(self.control_frame, width=120, height=10, bg="#17202A", fg="#AEDBCE", font=('Consolas', 11))
        self.text_area.pack(pady=10, padx=20)

    def get_color_hex(self, color_name):
        if color_name == 'Red': return '#E74C3C'
        if color_name == 'Blue': return '#2980B9'
        if color_name == 'Green': return '#27AE60'
        if color_name == 'Yellow': return '#F1C40F'
        return 'white'

    def draw_uno_card(self, parent, card, is_top_card=False, is_interactive=False):
        if is_top_card:
            w = 120
            h = 175
            center_size = 48
            corner_size = 20
        else:
            w = 75
            h = 110
            center_size = 32
            corner_size = 14
            
        bg_color = self.get_color_hex(card.color)
        c = tk.Canvas(parent, width=w, height=h, bg=bg_color, highlightthickness=4, highlightbackground="white")
        c.pack(side=tk.LEFT, padx=5, pady=5)
        
        pad_x = w * 0.15
        pad_y = h * 0.1
        c.create_oval(pad_x, pad_y, w - pad_x, h - pad_y, fill="white", outline="white")
        
        if card.value == 'Skip':
            val = "⊘"
        else:
            val = str(card.value)
        
        c.create_text((w/2)+2, (h/2)+2, text=val, font=('Arial', center_size, 'bold'), fill="black")
        c.create_text(w/2, h/2, text=val, font=('Arial', center_size, 'bold'), fill=bg_color)
        
        c.create_text((w*0.15)+1, (h*0.1)+1, text=val, font=('Arial', corner_size, 'bold'), fill="black")
        c.create_text(w*0.15, h*0.1, text=val, font=('Arial', corner_size, 'bold'), fill="white")
        
        c.create_text((w*0.85)+1, (h*0.9)+1, text=val, font=('Arial', corner_size, 'bold'), fill="black")
        c.create_text(w*0.85, h*0.9, text=val, font=('Arial', corner_size, 'bold'), fill="white")
        
        if is_interactive:
            c.bind("<Button-1>", lambda e, cd=card: self.user_clicked_card(cd))
            c.config(cursor="hand2")
            all_items = c.find_all()
            for item in all_items:
                c.tag_bind(item, "<Button-1>", lambda e, cd=card: self.user_clicked_card(cd))

    def update_ui(self):
        for frame in [self.p1_frame, self.p2_frame, self.p3_frame, self.top_card_container]:
            for widget in frame.winfo_children():
                widget.destroy()
                
        for card in self.state['p1_cards']: self.draw_uno_card(self.p1_frame, card)
        for card in self.state['p2_cards']: self.draw_uno_card(self.p2_frame, card)
        
        if not self.state['p1_cards']: tk.Label(self.p1_frame, text="WINNER!", font=('Arial', 16, 'bold'), bg="#2E4053", fg="gold").pack()
        if not self.state['p2_cards']: tk.Label(self.p2_frame, text="WINNER!", font=('Arial', 16, 'bold'), bg="#2E4053", fg="gold").pack()
        if not self.state['p3_cards']: tk.Label(self.p3_frame, text="WINNER!", font=('Arial', 16, 'bold'), bg="#2E4053", fg="gold").pack()

        p3_is_turn = (self.state['current_turn'] == 3)
        p3_interactive = (self.p3_mode == "MANUAL" and p3_is_turn and self.waiting_for_user)
        
        for card in self.state['p3_cards']: 
            self.draw_uno_card(self.p3_frame, card, False, p3_interactive)
            
        if self.p3_mode == "MANUAL":
            self.p3_frame.config(text="Player 3 (You)")
        else:
            self.p3_frame.config(text="Player 3 (Defensive)")
        
        self.draw_uno_card(self.top_card_container, self.state['top_card'], True)

        curr = self.state['current_turn']
        if curr == 1:
            self.p1_indicator.config(text="⬇️ CURRENT TURN ⬇️")
        else:
            self.p1_indicator.config(text="")
            
        if curr == 2:
            self.p2_indicator.config(text="⬇️ CURRENT TURN ⬇️")
        else:
            self.p2_indicator.config(text="")
            
        if curr == 3:
            self.p3_indicator.config(text="⬇️ CURRENT TURN ⬇️")
        else:
            self.p3_indicator.config(text="")

    def log(self, message):
        self.text_area.insert(tk.END, message + "\n")
        self.text_area.see(tk.END)

    def play_audio(self, sound_key, loop=False):
        if self.audio_enabled:
            if sound_key in self.sounds:
                if loop:
                    self.sounds[sound_key].play(loops=-1)
                else:
                    self.sounds[sound_key].play(loops=0)

    def user_clicked_card(self, card):
        if self.waiting_for_user:
            valid_moves = get_valid_moves(self.state['p3_cards'], self.state['top_card'])
            if card in valid_moves:
                self.process_move('p3_cards', card, 0)
            else:
                self.log("Invalid move!")

    def user_clicked_draw(self):
        if self.waiting_for_user:
            self.process_move('p3_cards', 'Draw', 0)

    def play_turn(self):
        game_over = is_terminal(self.state)
        if game_over or self.waiting_for_user:
            return

        self.uno_alert.config(text="") 
        turn = self.state['current_turn']
        
        if turn == 3 and self.p3_mode == "MANUAL":
            self.waiting_for_user = True
            self.log("\n--- Player 3's Turn (MANUAL) ---")
            self.log("It's your turn! Click a valid card or 'Draw Card' button.")
            self.next_btn.config(text="Draw Card", command=self.user_clicked_draw, bg="#3498DB", fg="white")
            self.update_ui()
            return

        # AI Turn Logic
        log_msg = "\n--- Player " + str(turn) + "'s Turn ---"
        self.log(log_msg)
        
        if turn == 1:
            player_key = 'p1_cards'
            res = minimax(self.state, 3, True, player_key)
            score = res[0]
            move = res[1]
        elif turn == 2:
            player_key = 'p2_cards'
            res = expectimax(self.state, 3, True, player_key)
            score = res[0]
            move = res[1]
        else: 
            player_key = 'p3_cards'
            res = minimax(self.state, 3, True, player_key)
            score = res[0]
            move = res[1]

        self.process_move(player_key, move, score)

    def process_move(self, player_key, move, score):
        turn = self.state['current_turn']
        
        if move == 'Draw':
            self.play_audio('draw')
        else:
            hand_len = len(self.state[player_key])
            if hand_len == 2:
                self.play_audio('uno')
                alert_msg = "PLAYER " + str(turn) + " YELLS UNO!"
                self.uno_alert.config(text=alert_msg)
                self.log("Player " + str(turn) + " yells UNO!")
                self.root.update()
                pygame.time.delay(500) 
            self.play_audio('play')

        if score != 0:
            score_text = "(Expected Score: " + str(score) + ")"
        else:
            score_text = "(Manual Input)"
            
        self.log("Chosen Move: " + str(move) + " " + score_text)
        
        self.state = apply_move(self.state, player_key, move)
        self.log("New Top Card: " + str(self.state['top_card']))
        
        self.waiting_for_user = False
        self.next_btn.config(text="Play Next Turn", command=self.play_turn, bg="#F39C12", fg="black")
        self.update_ui()
        
        if is_terminal(self.state):
            self.play_audio('win')
            if not self.state['p1_cards']:
                winner = "Player 1 (Defensive)"
            elif not self.state['p2_cards']:
                winner = "Player 2 (Offensive)"
            else:
                if self.p3_mode == "MANUAL":
                    winner = "Player 3 (You)"
                else:
                    winner = "Player 3 (Defensive)"
            
            self.p1_indicator.config(text="")
            self.p2_indicator.config(text="")
            self.p3_indicator.config(text="")
            
            self.log("\nGAME OVER! The winner is " + winner + "!")
            self.uno_alert.config(text="WINNER: " + winner + "!", fg="gold")
            self.next_btn.config(state=tk.DISABLED, text="Game Over")


demo_state = initialize_game()
generate_example_text_tree(demo_state, 'p1_cards')

root = tk.Tk()
app = UnoSimulationApp(root, demo_state)
root.mainloop()

->GENERATED SEARCH TREE(example)
Top card: Blue 3
AI hand:
Yellow 6
Red 3
Green 9
Red Skip
Yellow 2
->GENERATED Game Tree sim(depth=1)

AI decision(All possible decisions considered at this depth=1):

          AI (Root)
              |
Red 3   
              |
Opponent   

Play: Red 3
Expected score: 40.0
-



### Algorithm Comparison
#### 1. Strategies Used

Minimax (P1 and P3): These players played safe. They tried to keep special cards like "Skip" and assumed other players would always make the best move to stop them.

Expectimax (P2): This player was aggressive. It tried to get rid of cards quickly. It did not assume the worst; instead, it calculated the chance of getting a good card from the deck.

#### 2. Which One Won More?
Expectimax (Offensive) usually won faster. It reached 0 cards before the Minimax players in most tests.

#### 3. Why it Won
UNO has a lot of luck because of the draw pile. Minimax is too "scared" and thinks the game is always against it. This makes it hold onto cards too long. Expectimax understands that drawing a card is a random chance. By calculating those odds, it took better risks and emptied its hand much faster.